## Imports + connection

In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/checking-logs.sqlite")

## Query test set (Query one and returns two rows)

In [2]:
q_test = """
WITH base AS (
  SELECT
    t.uid,
    CASE
      WHEN julianday(t.first_commit_ts) < julianday(t.first_view_ts) THEN 'before'
      ELSE 'after'
    END AS time,
    (julianday(t.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0 AS diff_hours
  FROM test t
  JOIN deadlines d ON d.labs = t.labname
  WHERE t.labname <> 'project1'
),
eligible AS (
  SELECT uid
  FROM base
  GROUP BY uid
  HAVING SUM(time = 'before') > 0 AND SUM(time = 'after') > 0
)
SELECT
  time,
  AVG(diff_hours) AS avg_diff
FROM base
WHERE uid IN (SELECT uid FROM eligible)
GROUP BY time
ORDER BY time;
"""
test_results = pd.io.sql.read_sql(q_test, con)
test_results

,time,avg_diff
0,after,-105.229101
1,before,-61.156438


## Query control group (Query one and returns two rows)

In [3]:
q_control = """
WITH base AS (
  SELECT
    c.uid,
    CASE
      WHEN julianday(c.first_commit_ts) < julianday(c.first_view_ts) THEN 'before'
      ELSE 'after'
    END AS time,
    (julianday(c.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0 AS diff_hours
  FROM control c
  JOIN deadlines d ON d.labs = c.labname
  WHERE c.labname <> 'project1'
),
eligible AS (
  SELECT uid
  FROM base
  GROUP BY uid
  HAVING SUM(time = 'before') > 0 AND SUM(time = 'after') > 0
)
SELECT
  time,
  AVG(diff_hours) AS avg_diff
FROM base
WHERE uid IN (SELECT uid FROM eligible)
GROUP BY time
ORDER BY time;
"""
control_results = pd.io.sql.read_sql(q_control, con)
control_results

,time,avg_diff
0,after,-110.985496
1,before,-131.340449


In [ ]:
con.close()